# Estudo de Caso 4.1 — Solver Colebrook-White

**Capítulo Associado:** Capítulo 4 — Escoamento em Tubulações e Bombeamento  
**Nível:** Pós-Graduação  
**Tempo estimado:** 45 minutos  
**Autor:** Jader Lugon Junior  

---

## 📋 Resumo

Neste estudo de caso, você irá:

1. Implementar um solver iterativo para a equação de Colebrook-White
2. Comparar com a solução explícita de Swamee-Jain
3. Visualizar o Diagrama de Moody interativamente
4. Analisar a convergência do método de Newton-Raphson

---

## 🎯 Objetivos de Aprendizagem

- Resolver equações implícitas por métodos iterativos
- Avaliar a precisão de correlações explícitas
- Interpretar o Diagrama de Moody
- Implementar algoritmos numéricos robustos

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `scipy`, `matplotlib`
- Conhecimento prévio: Fator de atrito (Seção 4.3)

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from src import hidrodinamica

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Ambiente configurado com sucesso!")

## 🧮 A Equação de Colebrook-White

A equação de Colebrook-White é implícita no fator de atrito $f$:

$$\frac{1}{\sqrt{f}} = -2\log_{10}\left(\frac{\varepsilon/D}{3,7} + \frac{2,51}{\mathrm{Re}\sqrt{f}}\right)$$

Esta equação não pode ser resolvida algebricamente para $f$, exigindo métodos iterativos.

In [ ]:
# ============================================================
# IMPLEMENTAÇÃO DO SOLVER
# ============================================================

def colebrook_newton(Re, eps_D, tol=1e-8, max_iter=100):
    """
    Resolve Colebrook-White por Newton-Raphson.
    
    Define F(x) = x + 2*log10(eps_D/3.7 + 2.51/(Re*x)) = 0
    onde x = 1/sqrt(f)
    """
    # Estimativa inicial (Swamee-Jain)
    f = hidrodinamica.fator_atrito_swamee_jain(Re, eps_D)
    x = 1 / np.sqrt(f)
    
    historico = [f]
    
    for i in range(max_iter):
        # F(x)
        F = x + 2 * np.log10(eps_D/3.7 + 2.51/(Re*x))
        
        # F'(x)
        F_linha = 1 + 2/(np.log(10)) * (-2.51/(Re*x**2)) / (eps_D/3.7 + 2.51/(Re*x))
        
        # Newton-Raphson
        x_novo = x - F/F_linha
        
        # Critério de convergência
        if abs(x_novo - x) < tol:
            f_novo = 1/x_novo**2
            historico.append(f_novo)
            return f_novo, historico, i+1
        
        x = x_novo
        f = 1/x**2
        historico.append(f)
    
    raise RuntimeError(f"Colebrook não convergiu em {max_iter} iterações")

# Teste com exemplo
Re = 1e5
eps_D = 1e-4

f_newton, hist, iters = colebrook_newton(Re, eps_D)

print(f"📊 Exemplo: Re = {Re:.0e}, ε/D = {eps_D:.1e}")
print(f"  • f (Newton-Raphson) = {f_newton:.6f}")
print(f"  • Iterações: {iters}")
print(f"  • f (Swamee-Jain) = {hidrodinamica.fator_atrito_swamee_jain(Re, eps_D):.6f}")
print(f"  • Erro relativo: {abs(f_newton - hidrodinamica.fator_atrito_swamee_jain(Re, eps_D))/f_newton*100:.3f}%")

## 📈 Visualização da Convergência

In [ ]:
# ============================================================
# VISUALIZAÇÃO DA CONVERGÊNCIA
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(len(hist)), hist, 'bo-', linewidth=2, markersize=8)
ax.axhline(y=hist[-1], color='r', linestyle='--', linewidth=2, label=f'f_convergiu = {hist[-1]:.6f}')

ax.set_xlabel('Iteração', fontsize=12)
ax.set_ylabel('Fator de atrito, f', fontsize=12)
ax.set_title('Convergência do Método de Newton-Raphson', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\n💡 Interpretação:")
print(f"  O método de Newton-Raphson converge quadraticamente,")
print(f"  atingindo a solução em apenas {iters} iterações.")

## 🗺️ Diagrama de Moody Interativo

Vamos gerar o Diagrama de Moody completo.

In [ ]:
# ============================================================
# DIAGRAMA DE MOODY
# ============================================================

# Faixas de Reynolds e rugosidade relativa
Re_range = np.logspace(3, 8, 500)
eps_D_values = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 5e-2]

fig, ax = plt.subplots(figsize=(12, 8))

# Curvas para cada rugosidade relativa
for eps_D in eps_D_values:
    f_values = []
    for Re in Re_range:
        if Re < 2000:
            f_values.append(64/Re)
        else:
            try:
                f_values.append(hidrodinamica.fator_atrito_colebrook(Re, eps_D))
            except:
                f_values.append(np.nan)
    
    ax.loglog(Re_range, f_values, linewidth=2, label=f'ε/D = {eps_D:.0e}')

# Linha de regime laminar
Re_lam = np.logspace(3, 3.3, 100)
ax.loglog(Re_lam, 64/Re_lam, 'k-', linewidth=3, label='Laminar (f = 64/Re)')

# Linhas de transição
ax.axvline(x=2000, color='gray', linestyle=':', linewidth=2, alpha=0.5)
ax.axvline(x=4000, color='gray', linestyle=':', linewidth=2, alpha=0.5)
ax.text(2000, 0.1, 'Transição', rotation=90, fontsize=10, color='gray')

ax.set_xlabel('Número de Reynolds, Re', fontsize=12)
ax.set_ylabel('Fator de atrito, f', fontsize=12)
ax.set_title('Diagrama de Moody', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim([1e3, 1e8])
ax.set_ylim([0.008, 0.1])

plt.tight_layout()
plt.show()

## 🔍 Análise de Sensibilidade

Vamos explorar como $f$ varia com $\mathrm{Re}$ e $\varepsilon/D$.

In [ ]:
# ============================================================
# ANÁLISE DE SENSIBILIDADE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: f vs Re para diferentes ε/D
Re_range = np.logspace(4, 7, 100)
for eps_D in [1e-5, 1e-4, 1e-3, 1e-2]:
    f_values = [hidrodinamica.fator_atrito_colebrook(Re, eps_D) for Re in Re_range]
    axes[0].loglog(Re_range, f_values, linewidth=2, label=f'ε/D = {eps_D:.0e}')

axes[0].set_xlabel('Número de Reynolds, Re', fontsize=11)
axes[0].set_ylabel('Fator de atrito, f', fontsize=11)
axes[0].set_title('Efeito da Rugosidade Relativa', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Gráfico 2: f vs ε/D para diferentes Re
eps_D_range = np.logspace(-5, -2, 100)
for Re in [1e4, 1e5, 1e6, 1e7]:
    f_values = [hidrodinamica.fator_atrito_colebrook(Re, eps_D) for eps_D in eps_D_range]
    axes[1].loglog(eps_D_range, f_values, linewidth=2, label=f'Re = {Re:.0e}')

axes[1].set_xlabel('Rugosidade relativa, ε/D', fontsize=11)
axes[1].set_ylabel('Fator de atrito, f', fontsize=11)
axes[1].set_title('Efeito do Número de Reynolds', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Interpretação:")
print("   • Para Re altos (zona de resistência completa), f depende apenas de ε/D")
print("   • Para Re moderados, f depende de ambos (Re e ε/D)")
print("   • Para Re < 2000 (laminar), f = 64/Re (independente de ε/D)")

## 💡 Discussão

### Vantagens de Newton-Raphson

1. **Convergência rápida**: Tipicamente 4-5 iterações para precisão de 6 casas decimais
2. **Robustez**: Converge para qualquer combinação válida de Re e ε/D
3. **Precisão**: Resultado exato da equação de Colebrook-White

### Vantagens de Swamee-Jain

1. **Simplicidade**: Fórmula explícita, sem iteração
2. **Velocidade**: Cálculo direto, ideal para aplicações em tempo real
3. **Precisão aceitável**: Erro < 1% para a maioria das aplicações

### Recomendação

- **Projeto preliminar**: Use Swamee-Jain
- **Projeto detalhado**: Use Colebrook-White (Newton-Raphson)
- **Simulações em tempo real**: Use Swamee-Jain (mais rápido)

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 4](../notebooks/04_Escoamento_Tubulacoes.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 4.1**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>

In [ ]:
print("=" * 60)
print("✅ ESTUDO DE CASO CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Estudo de Caso 4.1!")
print("\n💡 Próximo passo: Explore o Estudo de Caso 4.2 (Bombeamento Predial)")